### Overview

When working with optical satellite imagery, we need to ensure the cloudy-pixels are removed from analysis. Most providers supply QA bands detailing locations of cloudy pixels. There are also third-party cloud-masking packages that can be used to locate and mask cloudy pixels.

In this notebook, we compare two cloud-masking approaches across a full time series of Sentinel-2 imagery: the Scene Classification (SCL) band supplied with Sentinel-2 Level-2A images, and [OmniCloudMask](https://github.com/DPIRD-DMA/OmniCloudMask), a deep-learning based cloud and cloud-shadow segmentation package. Both are applied to the same stack of scenes loaded from the STAC catalog, and compared via the NDVI time series they produce at a point of interest.

### Setup

Determine our runtime environment.

In [ ]:
import os

if 'COLAB_RELEASE_TAG' in os.environ:
    environment = 'colab'
    if os.environ.get('VERTEX_PRODUCT') == 'COLAB_ENTERPRISE':
        environment = 'colab_enterprise'
else:
    environment = 'local'

print(f'Environment: {environment}')

If we are on Google Colab, install the required packages. Local runtimes are expected to have the packages already installed.

In [ ]:
%%capture
if environment in ['colab', 'colab_enterprise']:
    !pip install pystac-client odc-stac rioxarray dask['distributed'] \
        jupyter-server-proxy odc-algo omnicloudmask

Import all required libraries. Make sure to import everything at the beginning as certain Xarray extensions are activated on import and registers certain accesors, like `.rio` and `.odc` for Xarray objects.

In [ ]:
import dask
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
import pystac_client
import rioxarray as rxr
import torch
import xarray as xr
from matplotlib.colors import ListedColormap
from odc.stac import configure_s3_access, load
from omnicloudmask import predict_from_array

Setup a local Dask cluster. This distributes the computation across multiple workers on your computer.

In [ ]:
from dask.distributed import Client

client = Client(
    n_workers=1,
    threads_per_worker=2,
    memory_limit='11GB',
)
client

If you are running this notebook in Colab, you will need to create and use a proxy URL to see the dashboard running on the local server.

In [ ]:
if environment == 'colab':
    from google.colab import output
    port_to_expose = 8787  # This is the default port for Dask dashboard
    print(output.eval_js(f'google.colab.kernel.proxyPort({port_to_expose})'))

### Location and STAC Catalog Setup


We define a location and time of interest, and a few constants that are reused throughout the notebook.

In [ ]:
latitude = 27.163
longitude = 82.608
year = 2023

Open a connection to the STAC catalog, and define the spectral bands and Sentinel-2 L2A scale/offset we'll use throughout &mdash; both the SCL and OmniCloudMask time series below are built from these same values.

In [ ]:
# Define a GeoJSON geometry
geometry = {
    'type': 'Point',
    'coordinates': [longitude, latitude]
}

# Query the STAC Catalog
catalog = pystac_client.Client.open(
    'https://earth-search.aws.element84.com/v1')

search = catalog.search(
    collections=['sentinel-2-c1-l2a'],
    intersects=geometry,
    datetime=f'{year}',
    query={
        'eo:cloud_cover': {'lt': 50},
        's2:nodata_pixel_percentage': {'lt': 10}},
    sortby=[
        {'field': 'properties.eo:cloud_cover',
         'direction': 'desc'}
        ]
)
items = search.item_collection()

ds = load(
    items,
    bands=['red', 'green', 'blue', 'nir', 'scl'],
    resolution=10, # Load the data at lower resolution to speed up processing
    crs='utm',
    chunks={'x': 1024, 'y': 1024},  # Explicitly define chunk sizes
    groupby='solar_day',
    preserve_original_order=True
)

# Apply scale/offset
# scale = 0.0001
# offset = -0.1
# # Select spectral bands (all except 'scl')
# data_bands = [band for band in ds.data_vars if band != 'scl']
# for band in data_bands:
#   ds[band] = ds[band] * scale + offset
ds

In [ ]:
# Items were sorted in descending order of cloud cover,
# so the first item is the most cloudy
most_cloudy = items[0]

scene = ds.sel(time=most_cloudy.datetime.replace(tzinfo=None))
scene = scene.squeeze()
scene

### Visualize the Scene

To visualize our Dataset, we first convert it to a DataArray using the `to_array()` method. All the variables will be converted to a new dimension. Since our variables are image bands, we give the name of the new dimesion as band.


In [ ]:
scene_da = scene.to_array('band')

In [ ]:
mask = scene['scl'].isin([3,8,9,10]).astype('uint8')

In [ ]:
scene_preview = scene_da.rio.reproject(
    scene_da.rio.crs, resolution=100
)
mask_preview = mask.rio.reproject(
    mask.rio.crs, resolution=100
)
fig, (ax0, ax1) = plt.subplots(1, 2)
fig.set_size_inches(10,5)
scene_preview.sel(band=['red', 'green', 'blue']).plot.imshow(
    ax=ax0,
    vmin=0, vmax=0.3)
ax0.set_title('RGB Visualization')

# RGBA: Transparent, Red
mask_colormap = ListedColormap(['#00000000', '#FF0000FF'])
mask_preview.plot.imshow(
    ax=ax1,
    cmap=mask_colormap,
    add_colorbar=False)

ax1.set_title('Cloud Mask')
for ax in (ax0, ax1):
  ax.set_axis_off()
  ax.set_aspect('equal')
plt.show()

#### OmniCloudMask across the time series

Unlike SCL, OmniCloudMask cannot be broadcast over `time`, it runs a model on one full scene at a time. A plain Python loop that calls `.compute()` for each date would work, but it breaks out of the lazy Dask pipeline and holds every mask in memory at once, which is exactly what we want to avoid when everything else here is lazy and Dask-backed.

Instead we use [`dask.array.map_blocks`](https://docs.dask.org/en/stable/generated/dask.array.core.map_blocks.html) on the underlying Dask array. This turns the per-scene prediction into a node in the Dask graph: the whole thing stays lazy, results are streamed rather than accumulated, and it composes directly with the composite step below. Because the model needs a whole scene at once, we make `y`, `x` and `band` single chunks and set `time` to one scene per chunk so each date is an independent task.

**How expensive is one scene?** OmniCloudMask runs a model per scene, so it is worth knowing the per-scene cost before scaling to a full stack. We pull the first date from the time-series cube and time three back-to-back predictions on it (the weights are already downloaded, so this measures the steady-state per-call cost).

If you are curious, `predict_from_array` re-instantiates its model on every call, so we also checked whether preloading the model once and reusing it via its `custom_models` argument would speed things up. On this runtime it made no measurable difference &mdash; the per-scene time is dominated by inference itself, not model setup &mdash; so we keep the simple call below rather than depend on an internal helper for no gain.

In [ ]:
%%time
inference_device = 'cuda' if torch.cuda.is_available() else 'cpu'
#inference_device = 'mps'
# Download/cache the model weights once here, in the main process.
# Without this, the first call to predict_from_array happens inside
# map_blocks, where multiple worker processes race to download the
# same weights file concurrently, causing a FileNotFoundError.
predict_from_array(
    np.zeros((3, 32, 32), dtype=np.float32), inference_device=inference_device
)

# OmniCloudMask needs the full spatial extent of a scene at once (it
# tiles internally), so y/x/band must be single chunks; we chunk
# `time` to 1 so each date becomes an independent task in the graph.
scene_da = scene_da.chunk(band=-1)

ocm_input = ds[['red', 'green', 'nir']] \
    .to_array('band') \
    .chunk({'time': 1, 'band': -1})


def _ocm_predict_block(arr):
  # arr arrives as (band, time=1, y, x) — to_array() prepends 'band'.
  scene_arr = np.nan_to_num(arr[:, 0], nan=0.0).astype(np.float32)
  pred = predict_from_array(scene_arr,  inference_device=inference_device)
  return (pred[0] > 0)[np.newaxis]  # (1, y, x) boolean: True where cloud/shadow

# map_blocks runs the per-scene prediction as a Dask graph node,
# so the whole pipeline stays lazy instead of computing in a loop.
ocm_mask_data = ocm_input.data.map_blocks(
    _ocm_predict_block,
    drop_axis=0,  # band axis is consumed by the prediction
    dtype=bool,
)
ocm_mask_ts = xr.DataArray(
    ocm_mask_data,
    dims=('time', 'y', 'x'),
    coords={'time': ocm_input.time, 'y': ocm_input.y, 'x': ocm_input.x},
).rio.write_crs(scene_da.rio.crs)  # Copy CRS from the input data

In [ ]:
scene_mask = ocm_mask_ts.sel(time=most_cloudy.datetime.replace(tzinfo=None))
scene_mask

In [ ]:
%%time
scene_mask = scene_mask.compute()

In [ ]:
scene_mask = scene_mask.astype('uint8')  # Convert boolean to uint8 for visualization

In [ ]:
scene_preview = scene_da.rio.reproject(
    scene_da.rio.crs, resolution=100
)
mask_preview = scene_mask.rio.write_crs(scene_da.rio.crs).rio.reproject(
    scene_mask.rio.crs, resolution=100
)
fig, (ax0, ax1) = plt.subplots(1, 2)
fig.set_size_inches(10,5)
scene_preview.sel(band=['red', 'green', 'blue']).plot.imshow(
    ax=ax0,
    vmin=0, vmax=3000)
ax0.set_title('RGB Visualization')

# RGBA: Transparent, Red
mask_colormap = ListedColormap(['#00000000', '#FF0000FF'])
mask_preview.plot.imshow(
    ax=ax1,
    cmap=mask_colormap,
    add_colorbar=False)

ax1.set_title('Cloud Mask')
for ax in (ax0, ax1):
  ax.set_axis_off()
  ax.set_aspect('equal')
plt.show()

In [ ]:
ts_masked_ocm = ds.where(~ocm_mask_ts)
ts_masked_ocm

In [ ]:
ts_mask = ds['scl'].isin([3, 8, 9, 10])
ts_masked_scl = ds[data_bands].where(~ts_mask)
ts_masked_scl

### Compare the Two Methods as a Time Series

A composite collapses the whole stack into one image, which hides *when* each method masked a pixel. The difference between the two cloud masks shows up most clearly in an analysis-ready **time series**: for a single location we can see which dates each mask drops and how noisy the retained signal is.

Following the course's [time-series workflow](https://courses.spatialthoughts.com/python-remote-sensing.html#extracting-and-processing-time-series), we extract an NDVI time series at a point of interest from each masked stack and compare them.

In [ ]:
# NDVI from each cloud-masked stack. Masked (cloudy) pixels are NaN,
# so they propagate as NaN into the NDVI time series.
def compute_ndvi(ds):
    return (ds['nir'] - ds['red']) / (ds['nir'] + ds['red'])

ndvi_scl = compute_ndvi(ts_masked_scl)
ndvi_ocm = compute_ndvi(ts_masked_ocm)

We reproject our point of interest into the cube's CRS and sample the nearest pixel across all dates, exactly as the course does with `interp(..., method='nearest')`. Dates whose pixel was flagged as cloud or shadow are `NaN`, so they appear as gaps in the series.

**Note:** the OmniCloudMask series forces per-scene model inference for every date. At the course's 10&nbsp;m resolution over a full year this is heavy, so expect this cell to run for a while (if it is too slow or runs out of memory, lower the load `resolution` in the datacube cell above).

In [ ]:
from pyproj import Transformer

# Reproject the point of interest (defined near the top of the
# notebook) from lat/lon into the cube's UTM CRS.
transformer = Transformer.from_crs(
    'EPSG:4326', ds.rio.crs, always_xy=True)
x0, y0 = transformer.transform(longitude, latitude)

# Sample the nearest pixel across every date, as the course does.
ndvi_scl_pt = ndvi_scl.interp(x=x0, y=y0, method='nearest').compute()
ndvi_ocm_pt = ndvi_ocm.interp(x=x0, y=y0, method='nearest').compute()

Plot the two raw series together. Wherever a line breaks, that date was masked out by that method &mdash; so the gaps themselves show where SCL and OmniCloudMask disagree about clouds at this location.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

ax.plot(ndvi_scl_pt.time.values, ndvi_scl_pt.values,
        '-o', ms=5, lw=2, color='#1e88e5', alpha=0.75,
        label='SCL masked', zorder=2)
ax.plot(ndvi_ocm_pt.time.values, ndvi_ocm_pt.values,
        '--^', ms=5, lw=1.5, color='#ff9900', alpha=0.75,
        label='OmniCloudMask masked', zorder=3)

ax.set_title('NDVI time series at the point of interest')
ax.set_xlabel('Date'); ax.set_ylabel('NDVI')
ax.set_ylim(-0.2, 1)
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

The raw series is gappy because cloudy dates are removed. Following the course's processing, we regularize it to a fixed 5-day cadence, linearly interpolate the interior gaps, back-/forward-fill the leading and trailing gaps, and apply a short centered rolling mean. Doing this for both masks shows how the choice of cloud mask propagates into the final smoothed curve.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

# Raw observations as faint markers (kept as-is, just lighter so the
# smoothed lines read clearly on top)
ax.plot(ndvi_scl_pt.time.values, ndvi_scl_pt.values,
        'o', ms=4, color='#1e88e5', alpha=0.25, zorder=1)
ax.plot(ndvi_ocm_pt.time.values, ndvi_ocm_pt.values,
        '^', ms=4, color='#ff9900', alpha=0.25, zorder=1)

# Smoothed series, styled so overlap is still visible as two lines
# ax.plot(ndvi_scl_smooth.time.values, ndvi_scl_smooth.values,
#         '-o', ms=3, lw=2, color='#1e88e5', alpha=0.75,
#         label='SCL (smoothed)', zorder=3)
# ax.plot(ndvi_ocm_smooth.time.values, ndvi_ocm_smooth.values,
#         '--^', ms=3, lw=1.5, color='#ff9900', alpha=0.75,
#         label='OmniCloudMask (smoothed)', zorder=2)

ax.set_title('Smoothed NDVI time series — SCL vs OmniCloudMask')
ax.set_xlabel('Date')
ax.set_ylabel('NDVI')
ax.set_ylim(-0.2, 1)
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

**A note on scaling up and on `map_blocks`:**

* We measured the per-scene cost and confirmed it is dominated by inference. `predict_from_array` does re-instantiate the model on each call, but with weights cached that overhead is negligible here, so reusing a preloaded model via `custom_models` gave no measurable speedup. The per-scene inference time is the real, unavoidable cost.
* **On a single GPU, be careful with parallelism.** `map_blocks` may run several time-chunks at once across workers, all competing for the same GPU and risking out-of-memory errors. For GPU inference, use a single worker (or a one-worker `Client`); reserve cross-worker parallelism for CPU.
* For a fully supported, file-based batch workflow, OmniCloudMask's own `predict_from_load_func` loads the weights once and batches inference across scenes.

Close the dask client. This presents multiple clients being instantiated when running different notebooks on the same machine. This is not required on Colab but a good practice when you are running it on a local machine. Uncomment and run to shutdown the dask cluster.

In [ ]:
#client.shutdown()